# NeuroSomno-HDL: Live Edge Inference on Zynq-7000 FPGA
This notebook demonstrates deploying the **NeuroSomno-HDL** sleep apnea detection accelerator on a **PYNQ-Z2** or **ZedBoard** FPGA using Python MMIO/DMA.

In [ ]:
import numpy as np
import time
from neurosomno_overlay import NeuroSomnoOverlay

# Initialize hardware accelerator
accel = NeuroSomnoOverlay(bitstream_path="neurosomno.bit")
accel.soft_reset()
print("[OK] Accelerator initialized and reset successfully.")

In [ ]:
# Load test data vectors
test_data = np.loadtxt("../../hardware/mif/test_data.txt", dtype=str)
test_labels = np.loadtxt("../../hardware/mif/test_labels.txt", dtype=str)

num_samples = len(test_labels)
print(f"Loaded {num_samples} pre-quantized test samples.")

In [ ]:
# Benchmark hardware execution latency
latencies = []
correct = 0

for i in range(num_samples):
    sample_784 = test_data[i * 784 : (i + 1) * 784]
    t0 = time.perf_counter()
    pred = accel.predict_sample(sample_784)
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1e6)
    
    act = int(test_labels[i], 2)
    if pred == act:
        correct += 1

print(f"Accuracy: {(correct / num_samples) * 100:.2f}%")
print(f"Mean Inference Latency: {np.mean(latencies):.2f} microseconds")
print(f"Clinical Telemetry: {accel.get_clinical_telemetry()}")